# Learning Objectives

In this notebook, you will craft sophisticated ETL jobs that interface with a variety of common data sources, such as 
- REST APIs (HTTP endpoints)
- RDBMS
- Hive tables (managed tables)
- Various file formats (csv, json, parquet, etc.)

d

# Interview Questions

As you progress through the practice, attempt to answer the following questions:

## Columnar File
- What is a columnar file format and what advantages does it offer?
- Why is Parquet frequently used with Spark and how does it function?
- How do you read/write data from/to a Parquet file using a DataFrame?

## Partitions
- How do you save data to a file system by partitions? (Hint: Provide the code)
- How and why can partitions reduce query execution time? (Hint: Give an example)

## JDBC and RDBMS
- How do you load data from an RDBMS into Spark? (Hint: Discuss the steps and JDBC)

## REST API and HTTP Requests
- How can Spark be used to fetch data from a REST API? (Hint: Discuss making API requests)

## ETL Job One: Parquet file
### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Data transformation requirements https://pgexercises.com/questions/aggregates/fachoursbymonth.html

### Load
Load data into a parquet file

### What is Parquet? 

Columnar files are an important technique for optimizing Spark queries. Additionally, they are often tested in interviews.
- https://www.youtube.com/watch?v=KLFadWdomyI
- https://www.databricks.com/glossary/what-is-parquet

In [0]:

from pyspark.sql.functions import *
from pyspark.sql.types import *

bookings = spark.table("bookings")
facilities = spark.table("facilities")
members = spark.table("members")

# Extract + Transform
job1 = bookings.filter((month("starttime") == 9) & (year("starttime") == 2012)).groupBy("facid").agg(sum("slots").alias("total_slots")).orderBy("facid")

job1.show()

# Load to parquet file
job1.write.mode("overwrite").parquet("/Volumes/workspace/default/csv_files/job1_parquet_output")


+-----+-----------+
|facid|total_slots|
+-----+-----------+
|    0|        591|
|    1|        588|
|    2|        570|
|    3|        422|
|    4|        648|
|    5|        122|
|    6|        540|
|    7|        426|
|    8|        471|
+-----+-----------+



## ETL Job Two: Partitions

### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Transform the data https://pgexercises.com/questions/joins/threejoin.html

### Load
Partition the result data by facility column and then save to `threejoin_delta` managed table. Additionally, they are often tested in interviews.

hint: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html

What are paritions? 

Partitions are an important technique to optimize Spark queries
- https://www.youtube.com/watch?v=hvF7tY2-L3U&t=268s

In [0]:
# How can you produce a list of all members who have used a tennis court? Include in your output the name of the court, and the name of the member formatted as a single column. Ensure no duplicate data, and order by the member name followed by the facility name.

from pyspark.sql.functions import *
from pyspark.sql.types import *

bookings = spark.table("bookings")
facilities = spark.table("facilities")
members = spark.table("members")

# Extract + Transform
job2 = bookings.join(members, bookings.memid == members.memid) \
    .join(facilities, bookings.facid == facilities.facid) \
    .select(
        concat(members.firstname, lit(" "), members.surname).alias("member_name"),
        facilities.name.alias("facility_name"),
        bookings.starttime,
        bookings.slots
    )

job2.show()

# Load as partitioned managed table
job2.write.mode("overwrite").partitionBy("facility_name").saveAsTable("threejoin_delta")


+------------+---------------+-------------------+-----+
| member_name|  facility_name|          starttime|slots|
+------------+---------------+-------------------+-----+
|Darren Smith|   Table Tennis|2012-07-03 11:00:00|    2|
|Darren Smith| Massage Room 1|2012-07-03 08:00:00|    2|
| GUEST GUEST|   Squash Court|2012-07-03 18:00:00|    2|
|Darren Smith|  Snooker Table|2012-07-03 19:00:00|    2|
|Darren Smith|     Pool Table|2012-07-03 10:00:00|    1|
|Darren Smith|     Pool Table|2012-07-03 15:00:00|    1|
| Tracy Smith| Tennis Court 1|2012-07-04 09:00:00|    3|
| Tracy Smith| Tennis Court 1|2012-07-04 15:00:00|    3|
|  Tim Rownam| Massage Room 1|2012-07-04 13:30:00|    2|
| GUEST GUEST| Massage Room 1|2012-07-04 15:00:00|    2|
| GUEST GUEST| Massage Room 1|2012-07-04 17:30:00|    2|
| GUEST GUEST|   Squash Court|2012-07-04 12:30:00|    2|
| GUEST GUEST|   Squash Court|2012-07-04 14:00:00|    2|
|Darren Smith|   Squash Court|2012-07-04 15:30:00|    2|
| Tracy Smith|  Snooker Table|2

## ETL Job Three: HTTP Requests

### Extract
Extract daily stock price data price from the following companies, Google, Apple, Microsoft, and Tesla. 

Data Source
- API: https://rapidapi.com/alphavantage/api/alpha-vantage
- Endpoint: GET `TIME_SERIES_DAILY`

Sample HTTP request

```
curl --request GET \
	--url 'https://alpha-vantage.p.rapidapi.com/query?function=TIME_SERIES_DAILY&symbol=TSLA&outputsize=compact&datatype=json' \
	--header 'X-RapidAPI-Host: alpha-vantage.p.rapidapi.com' \
	--header 'X-RapidAPI-Key: [YOUR_KEY]'

```

Sample Python HTTP request

```
import requests

url = "https://alpha-vantage.p.rapidapi.com/query"

querystring = {
    "function":"TIME_SERIES_DAILY",
    "symbol":"IBM",
    "datatype":"json",
    "outputsize":"compact"
}

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "[YOUR_KEY]"
}

response = requests.get(url, headers=headers, params=querystring)

data = response.json()

# Now 'data' contains the daily time series data for "IBM"
```

### Transform
Find **weekly** max closing price for each company.

hints: 
  - Use a `for-loop` to get stock data for each company
  - Use the spark `union` operation to concat all data into one DF
  - create a new `week` column from the data column
  - use `group by` to calcualte max closing price

### Load
- Partition `DF` by company
- Load the DF in to a managed table called, `max_closing_price_weekly`

In [0]:
import requests
import time
from pyspark.sql.functions import col, to_date, weekofyear, max

URL = "https://alpha-vantage.p.rapidapi.com/query"

HEADERS = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "cdbb794ca8msha0d0e0ca9a802e4p1d2170jsn5f7f62a7c64e"
}

SYMBOLS = ["GOOG", "AAPL", "MSFT", "TSLA"]

COLUMNS = ["symbol", "date", "close"]
final_df = None

# EXTRACT
for symbol in SYMBOLS:
    rows = []

    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": symbol,
        "outputsize": "compact",
        "datatype": "json"
    }

    response = requests.get(URL, headers=HEADERS, params=params)
    data = response.json()

    print("Checking:", symbol)

    if "Time Series (Daily)" in data:
        for dt, vals in data["Time Series (Daily)"].items():
            rows.append((
                symbol,
                dt,
                float(vals["4. close"])
            ))
    else:
        print("No data found for", symbol)
        print(data)

    if rows:
        current_df = spark.createDataFrame(rows, COLUMNS)

        if final_df is None:
            final_df = current_df
        else:
            final_df = final_df.union(current_df)

    time.sleep(1)

if final_df is None:
    raise ValueError("No data was fetched from the API.")

# TRANSFORM
transformed_df = final_df.withColumn("date", to_date(col("date"))) \
    .withColumn("week", weekofyear(col("date")))

weekly_max_df = transformed_df.groupBy("symbol", "week") \
    .agg(max("close").alias("max_closing_price")) \
    .orderBy("symbol", "week")
spark.sql("DROP TABLE IF EXISTS max_closing_price_weekly")
# LOAD
weekly_max_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("symbol") \
    .saveAsTable("max_closing_price_weekly")

# VERIFY
display(spark.table("max_closing_price_weekly"))

Checking: GOOG
Checking: AAPL
Checking: MSFT
Checking: TSLA


symbol,week,max_closing_price
AAPL,1,273.76
AAPL,2,267.26
AAPL,3,261.05
AAPL,4,248.35
AAPL,5,259.48
AAPL,6,278.12
AAPL,7,275.5
AAPL,8,264.58
AAPL,9,274.23
AAPL,10,264.72


## ETL Job Four: RDBMS


### Extract
Extract RNA data from a public PostgreSQL database.

- https://rnacentral.org/help/public-database
- Extract 100 RNA records from the `rna` table (hint: use `limit` in your sql)
- hint: use `spark.read.jdbc` https://docs.databricks.com/external-data/jdbc.html

### Transform
We want to load the data as it so there is no transformation required.


### Load
Load the DF in to a managed table called, `rna_100_records`

In [0]:

# ETL Job Four
print("Task 4")
# Extract
jdbc_url = "jdbc:postgresql://hh-pgsql-public.ebi.ac.uk:5432/pfmegrnargs"

connection_properties = {
    "user": "reader",
    "password": "NWDMCE5xdipIjRrp",
    "driver": "org.postgresql.Driver"
}

query = "(SELECT * FROM rna LIMIT 100) AS rna_subquery"

q4 = spark.read.jdbc(
    url=jdbc_url,
    table=query,
    properties=connection_properties
)

q4.show()
# transform : We want to load the data as it so there is no transformation required
# load
q4.write.mode("overwrite").saveAsTable("rna_100_records")


Task 4
+-------+-------------+-------------------+---------+----------------+----+--------------------+--------+--------------------+
|     id|          upi|          timestamp|userstamp|           crc64| len|           seq_short|seq_long|                 md5|
+-------+-------------+-------------------+---------+----------------+----+--------------------+--------+--------------------+
|3001278|URS00002DCBBE|2014-05-29 13:51:05|   RNACEN|0E653AC6FA32E6A0| 483|GAACGCTAGCGGCAGGC...|    NULL|7750e9f5890990af9...|
|3001279|URS00002DCBBF|2014-05-29 13:51:05|   RNACEN|A0CAF0D43413C685| 462|GAGTTTGATCATGGCTC...|    NULL|74f4a51608530e30f...|
|3001282|URS00002DCBC2|2014-05-29 13:51:05|   RNACEN|76D45E181DF9E92B|1737|TAGTCATATGCTTGTCT...|    NULL|7750ea0c27f7cfd8c...|
|3133418|URS00002FCFEA|2014-05-29 13:51:05|   RNACEN|167E572964FEF308|1318|GGATGAACGCTGGCGGT...|    NULL|d302791f198bbfcc5...|
|3133420|URS00002FCFEC|2014-05-29 13:51:05|   RNACEN|1B4FD6FAFA8005ED| 438|GCACAATGGGCGCAAGC...|    NULL